# 海馬迴 MRI 影像組學輔助內側顳葉癲癇側化（Colab 分析流程 v2）

對應計畫書第五段「分析流程」與切分圖。**順序是本 notebook 最重要的設計：**

| 模組 | 計畫書步驟 | 內容 | 是否使用跨受試者統計量 |
|---|---|---|---|
| M0 | — | 環境、參數、版本紀錄 | 否 |
| M1a | 第三段 | 臨床資料表篩選世代（去重、B8–B10），**不切分** | 否 |
| M2a | 步驟 0 | 逐人下載 OpenNeuro T1／FLAIR 與 figshare FreeSurfer（aseg、orig、aseg.stats） | 否 |
| M2b | 步驟 0 | 空間對齊檢查、兩套分割（aseg、HBT）一致性、海馬迴體積、分割品管縮圖（收／退／存疑） | 否 |
| M1b | 步驟 1 | 品管退件排除後**才切分**（70/30，seed 2026），測試集另存 | 否（只分層抽樣） |
| M2c | 步驟 0 | 前處理函式：N4、重新取樣、個人腦內 z 標準化 | 否 |
| M3 | 步驟 0 | PyRadiomics 萃取（原始／侵蝕／膨脹三版本遮罩）、左右不對稱指標 | 否 |
| M4 | 步驟 2–4 | **只用訓練集**：摺內 ComBat → ICC → 冗餘移除（以體積不對稱為參照）→ 標準化 → LASSO（≤ 5）→ 擬合；選超參數、模型 C 的巢狀 CV、定 Youden 切點、定版 | 是，但全部在摺內 |
| M5 | 步驟 5–6 | **保留測試集只開啟一次**：AUC、AUPRC、校準、決策曲線、模型 C vs A 的 ΔAUC | — |

**兩種模式**：`DEMO_MODE = True`（預設）時，M3 以 PyRadiomics 內建示範影像驗證萃取可執行，M4–M5 以模擬特徵表演練完整流程，**示範數字不是研究結果**。改成 `False` 前，請到 cnnp-lab.com/ideas-data 查 IDEAS 的 OpenNeuro 編號，並依 `participants.tsv` 實際欄位修改 `COLS`。

**計畫書補充說明（2026-11-25，開啟測試集前寫定）**：B1 冗餘移除以 `vol_ai` 為參照；B2 模型 C 的訓練集 CV AUC 以巢狀 CV 估計；B3 摺內順序為 ComBat → ICC；B4 ICC(3,1) 於不對稱指標層級計算。2026-11-27 追加 B11：ICC 改以兩套獨立演算法的分割（FreeSurfer aseg 與海馬迴亞區模組 HBT）計算；侵蝕／膨脹擾動保留為探索性分析。詳見 `docs/計畫書補充說明_20261125.md`。

## M0｜環境建置、事先寫定的參數與版本紀錄

In [ ]:
# 版本鎖定（2026-11-25 於 Colab 驗證）：PyPI 版 PyRadiomics 在 Python 3.13 無法編譯，改裝 GitHub 上的固定版本
!pip -q install "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics.git@8ed579383b44806651c463d5e691f3b2b57522ab"
!pip -q install neuroCombat==0.2.12 SimpleITK==2.5.8 nibabel openneuro-py
# numpy、pandas、scipy、scikit-learn 使用 Colab 內建版本，實際版本由下一格寫入 versions.json；與 requirements.txt 不同時請在報告中註明


In [ ]:
import os, json, warnings, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import stats
from scipy.optimize import brentq
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore")

DEMO_MODE = True
SEED = 2026
np.random.seed(SEED); random.seed(SEED)

# ---- 事先寫定（計畫書第四段），分析時不再擴充 ----
TEST_SIZE   = 0.30
ICC_MIN     = 0.75
CORR_MAX    = 0.90
K_MAX       = 5                                  # 訓練集事件 59 ÷ 10，無條件捨去
LASSO_GRID  = np.logspace(-3, 1, 30)             # 30 點
RF_GRID     = {"n_estimators": [300, 500, 1000], "max_depth": [3, 5, None]}   # 9 組
CV          = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
N_BOOT      = 1000
DELTA_AUC_H = 0.05                               # 主要假說：C 比 A 提升 ≥ 0.05
MAX_GAP     = 0.05                               # 主要假說第二條：C 的訓練集 CV AUC − 測試 AUC < 0.05（B7：單向）
ALPHA       = 0.05                               # DeLong 檢定顯著水準

# ---- 計畫書補充說明（2026-11-25，開啟測試集前寫定；docs/計畫書補充說明_20261125.md）----
REDUNDANCY_REF = "vol_ai"                        # B1：冗餘移除以模型 A 的體積不對稱指標為已保留的參照
INNER_CV    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)   # B2：模型 C 巢狀 CV 的內層（外層沿用 CV）

PILOT_N    = None                                # 試跑時設成 3：M1a 後只取前 3 人；試跑模式不執行 M1b 切分
HBT_HIPPO  = [226, 231, 232]                     # B11：海馬迴亞區模組 HBT 檔中的海馬迴尾、體、頭（不含海馬裂 215 與杏仁核）
DATASET_ID = "ds005602"                          # IDEAS 的 OpenNeuro 編號（v1.0.0，2024-10-28；僅含原始 T1w、FLAIR）
# 注意：FreeSurfer 分割、臨床資料表、切除遮罩不在 OpenNeuro，而在 figshare（連結見 OpenNeuro README 與 cnnp-lab.com/ideas-data）；M1 下載段待確認檔案結構後改寫
ROOT = "/content/drive/MyDrive/ideas_radiomics" if not DEMO_MODE else "/content/demo_radiomics"
if not DEMO_MODE:
    # 先掛載 Google Drive 再建立資料夾；若先建資料夾，/content/drive 會變成本機資料夾而無法掛載
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(ROOT, exist_ok=True)
# 臨床欄位依 figshare tables_metadata（readme_metadata.pdf）：ID、Op_Side、Op_Type、Pathology、Binned_Age_at_Surgery、Sex、ILAE_Year1…
ASEG = {"hip_L": 17, "hip_R": 53, "amy_L": 18, "amy_R": 54}

import sklearn, scipy, sys
from importlib.metadata import version as _ver, PackageNotFoundError
versions = {"python_seed": SEED, "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "SimpleITK": sitk.Version_VersionString()}
for pkg in ["pyradiomics", "neuroCombat"]:
    try: versions[pkg] = _ver(pkg)
    except PackageNotFoundError: versions[pkg] = "not installed"
json.dump(versions, open(os.path.join(ROOT, "versions.json"), "w"), indent=1)
print(versions)

## M1a｜整理世代與納入排除（計畫書第三段；補充說明 B8–B10）

只用臨床資料表與檔案是否存在來篩選，不看任何影像內容。原始臨床資料表不修改，所有處理在程式內完成並記入排除流程。**本段不切分**：依計畫書，分割品管排除要在切分之前，所以切分移到 M1b。

In [ ]:
import time, requests
from concurrent.futures import ThreadPoolExecutor
S3 = f"https://s3.amazonaws.com/openneuro.org/{DATASET_ID}"          # OpenNeuro 的公開 S3 路徑
def s3_url(i, suffix): return f"{S3}/sub-{i}/anat/sub-{i}_{suffix}"
def s3_exists(url):
    for attempt in range(3):
        try: return requests.head(url, timeout=30).status_code == 200
        except requests.RequestException: time.sleep(3)
    raise RuntimeError(f"無法連線：{url}")

if not DEMO_MODE:
    META_DIR = os.path.join(ROOT, "metadata")                       # 放 figshare tables_metadata 的三個檔
    meta = pd.read_csv(os.path.join(META_DIR, "Metadata_Release_Anon.csv"))
    flow = [("IDEAS 病人資料列", len(meta))]
    meta = meta.drop_duplicates(); flow.append(("去除完全重複的資料列（原始表 ID 209 重複一列）", len(meta)))
    assert meta["ID"].is_unique, "去除完全重複列後仍有重複 ID，需人工檢查"
    meta = meta[meta["Op_Type"] == "T Lx"]; flow.append(("單側顳葉切除（Op_Type = T Lx；排除病灶切除、多腦葉、顳葉外）", len(meta)))
    meta = meta[meta["Pathology"].isin(["HS", "OTHER"])]; flow.append(("病理為 HS 或非病灶性 OTHER（B9）", len(meta)))
    with ThreadPoolExecutor(16) as ex:
        has_t1 = dict(zip(meta["ID"], ex.map(lambda i: s3_exists(s3_url(i, "T1w.nii.gz")), meta["ID"])))
        has_fl = dict(zip(meta["ID"], ex.map(lambda i: s3_exists(s3_url(i, "FLAIR.nii.gz")), meta["ID"])))
    meta = meta[meta["ID"].map(has_t1)]; flow.append(("具術前 3D T1（OpenNeuro）", len(meta)))
    meta = meta[meta["ID"].map(has_fl)]; flow.append(("具 3D FLAIR（B10）", len(meta)))
    # B8：年齡級距全部納入，不另排除
    cohort = meta.assign(pid=meta["ID"].map(lambda i: f"sub-{i}"), y=(meta["Op_Side"] == "R").astype(int)).set_index("pid")
    print("（2026-11-27 以臨床資料表與 OpenNeuro 檔案清單試算的預期值：316 → 257 → 257 → 244）")
else:
    # 示範：與計畫書預估相同的人數與側別分布（左 122、右 85）
    rng = np.random.default_rng(SEED)
    n_L, n_R = 122, 85
    cohort = pd.DataFrame({"y": np.r_[np.zeros(n_L), np.ones(n_R)].astype(int)},
                          index=[f"sub-{i:03d}" for i in range(n_L + n_R)])
    flow = [("示範世代", len(cohort))]
print(pd.DataFrame(flow, columns=["步驟", "剩餘人數"]).to_string(index=False))
if PILOT_N:
    cohort = cohort.iloc[:PILOT_N]; print(f"試跑模式：只取前 {PILOT_N} 人（不執行 M1b 切分）")
y = cohort["y"]; print(f"世代：n = {len(cohort)}，左 {int((y == 0).sum())}，右 {int(y.sum())}")


## M2a｜逐人下載與格式轉換（逐一受試者，切分前允許）

每位受試者存到 Drive 的 `subjects/sub-<ID>/`：
- OpenNeuro：`T1w.nii.gz`、`FLAIR.nii.gz` 與 JSON
- figshare FreeSurfer：只從 `<ID>.tar.gz` 取出 `mri/aseg.mgz`、`mri/orig.mgz`、`stats/aseg.stats`，以及海馬迴亞區模組的 `mri/lh.hippoAmygLabels.HBT.FSvoxelSpace.mgz`、`mri/rh.hippoAmygLabels.HBT.FSvoxelSpace.mgz`（B11，第二套分割），影像都轉成 NIfTI；壓縮檔核對 MD5 後即刪除

已完成且檔案齊全者自動跳過；B11 之前試跑過、缺 HBT 檔的人會自動重新下載。244 人約需下載 50 GB（逐人下載後即刪除壓縮檔），Drive 端約佔 4–5 GB。

In [ ]:
import hashlib, tarfile, shutil
import nibabel as nib
FS_KEY = "b13b8bb41390d3f7a088"                                     # figshare freesurfer_zip 的分享代碼（公開於 OpenNeuro README）
FS_LIST_URL = "https://raw.githubusercontent.com/daniel-ndmu/radiomics-hippocampus-tle/main/config/figshare_freesurfer_files.csv"
SUBJ, TMP = os.path.join(ROOT, "subjects"), "/content/tmp_fs"
KEEP = ["mri/aseg.mgz", "mri/orig.mgz", "stats/aseg.stats",
        "mri/lh.hippoAmygLabels.HBT.FSvoxelSpace.mgz", "mri/rh.hippoAmygLabels.HBT.FSvoxelSpace.mgz"]   # 後兩者：B11 第二套分割
OUT_FILES = ["T1w.nii.gz", "FLAIR.nii.gz", "aseg.nii.gz", "orig.nii.gz", "aseg.stats", "hbt_lh.nii.gz", "hbt_rh.nii.gz"]

def download(url, dest, md5=None, chunk=8 << 20):
    for attempt in range(3):
        try:
            h = hashlib.md5()
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                if r.status_code != 200 or "text/html" in r.headers.get("Content-Type", ""):
                    raise requests.RequestException(f"伺服器回傳 {r.status_code}（{r.headers.get('Content-Type')}），不是檔案本體")
                with open(dest, "wb") as f:
                    for b in r.iter_content(chunk): f.write(b); h.update(b)
            if md5 is None or h.hexdigest() == md5: return
            print(f"MD5 不符，重新下載：{url}")
        except requests.RequestException as e:
            print(f"下載失敗（第 {attempt + 1} 次）：{e}"); time.sleep(10)
    raise RuntimeError(f"下載失敗：{url}")

def mgz_to_nii(src, dst):
    """FreeSurfer .mgz → NIfTI，保留原本的空間座標（affine），SimpleITK 才讀得到"""
    img = nib.load(src); nib.save(nib.Nifti1Image(np.asanyarray(img.dataobj), img.affine), dst)

def fetch_subject(i):
    d = os.path.join(SUBJ, f"sub-{i}")
    if os.path.exists(os.path.join(d, ".done")) and all(os.path.exists(os.path.join(d, f)) for f in OUT_FILES):
        return i, "已完成，略過"
    try:
        os.makedirs(d, exist_ok=True)
        for suffix in ["T1w.nii.gz", "T1w.json", "FLAIR.nii.gz", "FLAIR.json"]:
            download(s3_url(i, suffix), os.path.join(d, suffix))
        row = fs_files.loc[f"{i}.tar.gz"]
        tgz = os.path.join(TMP, f"{i}.tar.gz")
        # 2026-11-27 Colab 實測：figshare.com/ndownloader 回 202（驗證頁），ndownloader.figshare.com 會 302 轉到 S3 取得檔案本體
        download(f"https://ndownloader.figshare.com/files/{row.file_id}?private_link={FS_KEY}", tgz, md5=row.md5)
        with tarfile.open(tgz, "r:gz") as t:
            members = {m.name.split("/", 1)[1]: m for m in t.getmembers()
                       if m.isfile() and m.name.split("/", 1)[0] == str(i) and "/" in m.name}
            missing = [k for k in KEEP if k not in members]
            if missing: raise RuntimeError(f"壓縮檔缺少 {missing}")
            for k in KEEP:
                with t.extractfile(members[k]) as src, open(os.path.join(TMP, f"{i}_{os.path.basename(k)}"), "wb") as dst:
                    shutil.copyfileobj(src, dst)
        os.remove(tgz)
        conv = {"aseg.mgz": "aseg.nii.gz", "orig.mgz": "orig.nii.gz",
                "lh.hippoAmygLabels.HBT.FSvoxelSpace.mgz": "hbt_lh.nii.gz", "rh.hippoAmygLabels.HBT.FSvoxelSpace.mgz": "hbt_rh.nii.gz"}
        for src, dst in conv.items():
            mgz_to_nii(os.path.join(TMP, f"{i}_{src}"), os.path.join(d, dst)); os.remove(os.path.join(TMP, f"{i}_{src}"))
        shutil.move(os.path.join(TMP, f"{i}_aseg.stats"), os.path.join(d, "aseg.stats"))
        open(os.path.join(d, ".done"), "w").close()
        return i, "完成"
    except Exception as e:
        return i, f"錯誤：{e}"

if not DEMO_MODE:
    os.makedirs(SUBJ, exist_ok=True); os.makedirs(TMP, exist_ok=True)
    fs_files = pd.read_csv(FS_LIST_URL).set_index("name")
    assert len(fs_files) == 542 and fs_files.loc["1.tar.gz", "file_id"] == 45685371, "figshare 檔案清單與 2026-11-27 核對的版本不符"
    ids = cohort["ID"].tolist(); t0 = time.time(); log = []
    print(f"本機暫存空間剩餘 {shutil.disk_usage('/content').free / 2**30:.0f} GiB；開始處理 {len(ids)} 人")
    with ThreadPoolExecutor(4) as ex:
        for k, (i, status) in enumerate(ex.map(fetch_subject, ids), 1):
            log.append((i, status))
            if status.startswith("錯誤") or k % 10 == 0:
                print(f"[{k}/{len(ids)}，{(time.time() - t0) / 60:.0f} 分鐘] sub-{i}：{status}")
    log = pd.DataFrame(log, columns=["ID", "status"]); log.to_csv(os.path.join(ROOT, "fetch_log.csv"), index=False)
    bad = log[log["status"].str.startswith("錯誤")]
    print(f"完成 {len(log) - len(bad)} 人；錯誤 {len(bad)} 人" + ("（重新執行本格即可續傳）" if len(bad) else ""))
else:
    print("示範模式：略過 M2a（不下載真實資料）")


## M2b｜空間對齊檢查、海馬迴體積與分割品管（逐一受試者，切分前允許）

1. **對齊檢查**：FreeSurfer 的 `orig` 是由原始 T1 轉換而來，依實際空間座標重新取樣到原始 T1 後，兩者在腦內的灰階相關應該很高。相關係數偏低代表座標對不上，標記出來優先檢查。
2. **海馬迴體積**：由 `aseg.stats` 讀出左右海馬迴與杏仁核體積、eTIV，算出體積不對稱指標 `vol_ai`（模型 A 用）。
3. **兩套分割的一致性（B11）**：FreeSurfer aseg 與海馬迴亞區模組 HBT 的海馬迴，逐側計算 Dice；Dice 偏低者標記。
4. **品管縮圖**：每頁 20 人，每人 3 張冠狀切面（後、中、前段），aseg 以實線、HBT 以虛線疊在 FreeSurfer T1 上。**縮圖不顯示切除側別與病理**，避免判讀時受影響。兩套分割都要合理才判「收」。

判讀後，把 `qc/qc_decisions_template.csv` 另存為 `qc_decisions.csv`（放在 `ROOT`），`decision` 欄填 **收**、**退** 或 **存疑**（W11 判準；本專題不修改 IDEAS 的分割，所以沒有「修」）。存疑者須在切分前改判為收或退。

In [ ]:
def parse_aseg_stats(path):
    """讀 FreeSurfer aseg.stats：eTIV 與各結構體積（mm³）"""
    out = {}
    for line in open(path):
        if line.startswith("# Measure EstimatedTotalIntraCranialVol"):
            out["eTIV"] = float(line.split(",")[3])
        elif not line.startswith("#"):
            p = line.split()
            if len(p) >= 5 and p[4] in ("Left-Hippocampus", "Right-Hippocampus", "Left-Amygdala", "Right-Amygdala"):
                out[p[4]] = float(p[3])
    return out

def canon(path):
    img = nib.as_closest_canonical(nib.load(path)); return np.asanyarray(img.dataobj)

def hbt_label_image(d, ref=None):
    """B11：把左右 HBT 檔合成一個與 aseg 同編碼的標籤影像（左 17、右 53），只含海馬迴尾、體、頭"""
    out = None
    for side, lab in [("lh", 17), ("rh", 53)]:
        h = sitk.ReadImage(os.path.join(d, f"hbt_{side}.nii.gz"))
        a = sitk.GetArrayFromImage(h)
        if not np.isin(a, HBT_HIPPO).any():
            raise RuntimeError(f"{d} 的 hbt_{side} 找不到海馬迴標籤 {HBT_HIPPO}；實際標籤：{np.unique(a)[:30]}")
        m = sitk.GetImageFromArray(np.where(np.isin(a, HBT_HIPPO), lab, 0).astype(np.uint8)); m.CopyInformation(h)
        out = m if out is None else sitk.Maximum(out, sitk.Resample(m, out, sitk.Transform(), sitk.sitkNearestNeighbor, 0))
    return out if ref is None else sitk.Resample(out, ref, sitk.Transform(), sitk.sitkNearestNeighbor, 0)

def dice(a, b): return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

def qc_row(axes, i):
    d = os.path.join(SUBJ, f"sub-{i}")
    t1, seg = canon(os.path.join(d, "orig.nii.gz")).astype(float), canon(os.path.join(d, "aseg.nii.gz")).astype(int)
    hbt = np.zeros_like(seg)
    for side, lab in [("lh", 17), ("rh", 53)]:
        hbt[np.isin(canon(os.path.join(d, f"hbt_{side}.nii.gz")).astype(int), HBT_HIPPO)] = lab
    hip = np.isin(seg, [17, 53])
    xs, ys, zs = np.where(hip)
    for ax, frac in zip(axes, [0.25, 0.5, 0.75]):
        j = int(np.percentile(ys, frac * 100))                       # RAS：第二軸為前後，取冠狀切面
        sl, sg = t1[:, j, :].T, seg[:, j, :].T
        x0, x1, z0, z1 = xs.min() - 8, xs.max() + 8, zs.min() - 15, zs.max() + 15
        ax.imshow(sl[z0:z1, x0:x1], cmap="gray", origin="lower")
        hb = hbt[:, j, :].T
        for lab, col in [(17, "red"), (53, "cyan")]:
            m = (sg == lab)[z0:z1, x0:x1]
            if m.any(): ax.contour(m, levels=[0.5], colors=col, linewidths=0.6)
            m2 = (hb == lab)[z0:z1, x0:x1]
            if m2.any(): ax.contour(m2, levels=[0.5], colors="yellow", linewidths=0.6, linestyles="dashed")
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(f"sub-{i}", fontsize=8)

if not DEMO_MODE:
    rows = []
    for i in cohort["ID"]:
        d = os.path.join(SUBJ, f"sub-{i}")
        t1 = sitk.ReadImage(os.path.join(d, "T1w.nii.gz"), sitk.sitkFloat32)
        aseg = sitk.Resample(sitk.ReadImage(os.path.join(d, "aseg.nii.gz")), t1, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
        orig = sitk.Resample(sitk.ReadImage(os.path.join(d, "orig.nii.gz"), sitk.sitkFloat32), t1, sitk.Transform(), sitk.sitkLinear, 0)
        a, o, m = (sitk.GetArrayFromImage(x) for x in (t1, orig, aseg))
        brain = m > 0
        r = np.corrcoef(a[brain], o[brain])[0, 1] if brain.sum() > 1000 else np.nan
        vox = np.prod(t1.GetSpacing())
        st = parse_aseg_stats(os.path.join(d, "aseg.stats"))
        L, R = st["Left-Hippocampus"], st["Right-Hippocampus"]
        hb = sitk.GetArrayFromImage(hbt_label_image(d, ref=t1))
        rows.append({"pid": f"sub-{i}", "ID": i, "align_r": r,
                     "dice_aseg_hbt_L": dice(m == 17, hb == 17), "dice_aseg_hbt_R": dice(m == 53, hb == 53),
                     "hipL_native_over_stats": (m == 17).sum() * vox / L, "hipR_native_over_stats": (m == 53).sum() * vox / R,
                     "eTIV": st["eTIV"], "hip_L": L, "hip_R": R, "amy_L": st["Left-Amygdala"], "amy_R": st["Right-Amygdala"],
                     "vol_ai": (L - R) / ((L + R) / 2)})
    qcm = pd.DataFrame(rows).set_index("pid")
    qcm["flag_alignment"] = (qcm["align_r"] < 0.8) | ~qcm[["hipL_native_over_stats", "hipR_native_over_stats"]].apply(lambda s: s.between(0.8, 1.2)).all(axis=1)
    qcm["flag_seg_disagree"] = qcm[["dice_aseg_hbt_L", "dice_aseg_hbt_R"]].min(axis=1) < 0.7   # 兩套分割明顯不一致
    qcm.to_csv(os.path.join(ROOT, "volumes_alignment.csv"))
    print(qcm[["align_r", "hipL_native_over_stats", "hipR_native_over_stats", "dice_aseg_hbt_L", "dice_aseg_hbt_R"]].describe().round(3))
    print("對齊需優先檢查：", qcm.index[qcm["flag_alignment"]].tolist())
    print("兩套分割明顯不一致（Dice < 0.7）：", qcm.index[qcm["flag_seg_disagree"]].tolist())

    # 品管縮圖：依 ID 排序、每頁 20 人；不顯示側別與病理
    QC = os.path.join(ROOT, "qc"); os.makedirs(QC, exist_ok=True)
    ids = sorted(cohort["ID"])
    for p in range(0, len(ids), 20):
        chunk = ids[p:p + 20]
        fig, axs = plt.subplots(len(chunk), 3, figsize=(6, 1.6 * len(chunk)))
        axs = np.atleast_2d(axs)
        for row_axes, i in zip(axs, chunk): qc_row(row_axes, i)
        for ax, t in zip(axs[0], ["posterior (tail)", "middle (body)", "anterior (head)"]): ax.set_title(t, fontsize=8)
        fig.suptitle("Hippocampus QC, coronal, neurological view (image left = subject left)\n"
                     "solid: FreeSurfer aseg (red = left 17, cyan = right 53); yellow dashed: hippocampal subfield HBT", fontsize=8)
        plt.tight_layout(); fig.savefig(os.path.join(QC, f"qc_page_{p // 20 + 1:02d}.png"), dpi=150); plt.close(fig)
    tmpl = qcm[["ID", "align_r", "flag_alignment", "dice_aseg_hbt_L", "dice_aseg_hbt_R", "flag_seg_disagree"]].assign(decision="", note="")
    tmpl.sort_values("ID").to_csv(os.path.join(QC, "qc_decisions_template.csv"))
    # 判讀者內一致性（W11 第 ⑤ 步的折衷）：隨機抽 10% 供一週後盲審第二次
    pd.Series(np.random.default_rng(SEED).choice(tmpl.index, size=max(1, len(tmpl) // 10), replace=False),
              name="pid").to_csv(os.path.join(QC, "qc_second_review_sample.csv"), index=False)
    print(f"品管縮圖 {int(np.ceil(len(ids) / 20))} 頁已存到 {QC}；判讀後存成 {os.path.join(ROOT, 'qc_decisions.csv')}")
else:
    print("示範模式：略過 M2b")


## M1b｜品管後切分（計畫書步驟 1）

依計畫書，分割品管不通過者先排除，再以病人為單位、依側別分層切出 30% 保留測試集（seed 2026）。切分只用側別，不看任何影像特徵。測試集名單寫入 `test_ids.csv` 後，M4 完全不讀取測試集的特徵。

In [ ]:
if PILOT_N:
    raise SystemExit("試跑模式（PILOT_N）不切分：請把 PILOT_N 改回 None、跑完全部受試者並完成品管後再執行本格。")
if not DEMO_MODE:
    qc_path = os.path.join(ROOT, "qc_decisions.csv")
    if not os.path.exists(qc_path):
        raise SystemExit("尚未完成 FreeSurfer 分割品管：請先依 M2b 判讀並存成 qc_decisions.csv，再執行本格。")
    qc = pd.read_csv(qc_path, index_col="pid")["decision"].reindex(cohort.index)
    assert qc.notna().all(), "有受試者尚未品管"
    assert qc.isin(["收", "退"]).all(), "decision 只能是「收」或「退」；「存疑」須在切分前判定"
    cohort = cohort[qc.eq("收")]; flow.append(("FreeSurfer 海馬迴分割品管通過（退者排除）", len(cohort)))

train_ids, test_ids = train_test_split(cohort.index, test_size=TEST_SIZE, stratify=cohort["y"], random_state=SEED)
pd.Series(test_ids).to_csv(os.path.join(ROOT, "test_ids.csv"), index=False)
for name, ids in [("全部", cohort.index), ("訓練集", train_ids), ("保留測試集", test_ids)]:
    y = cohort.loc[ids, "y"]
    print(f"{name}: n = {len(ids)}，左 {int((y == 0).sum())}，右（事件）{int(y.sum())}")
print(pd.DataFrame(flow, columns=["步驟", "剩餘人數"]).to_string(index=False))
pd.DataFrame(flow, columns=["step", "n"]).to_csv(os.path.join(ROOT, "flow.csv"), index=False)


## M2c｜逐一受試者前處理函式（計畫書步驟 0；M3 萃取時呼叫）

每個運算只用該受試者自己的影像：N4 偏場校正 → B-spline 重新取樣至 1 mm → 以自己的腦組織平均與標準差做 z 分數（截斷 ±3）。遮罩一律最近鄰內插。

FLAIR 先以剛體配準對到 T1（`register_flair_to_t1`），再把 FreeSurfer 分割以反轉換、最近鄰內插帶到 FLAIR 的影像空間（計畫書第四段「遮罩以最近鄰內插轉換到 T1 與 FLAIR 之影像空間」）。每人的轉換存成 `flair_to_t1.tfm`，平移與旋轉量記入 `flair_registration.csv` 供檢查。


In [ ]:
def n4(img, mask, shrink=4):
    """N4 偏場校正：在縮小 shrink 倍的影像上估計偏場，再以 GetLogBiasFieldAsImage 套回原解析度。
    這是 SimpleITK 官方範例的做法；偏場本身是平滑的低頻場，結果與全解析度估計相近，速度快數十倍
    （2026-11-27 Colab 實測：全解析度估計 3 人逾 30 分鐘仍未完成）。"""
    img = sitk.Cast(img, sitk.sitkFloat32); mask = sitk.Cast(mask, sitk.sitkUInt8)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.Execute(sitk.Shrink(img, [shrink] * 3), sitk.Shrink(mask, [shrink] * 3))
    return img / sitk.Exp(corrector.GetLogBiasFieldAsImage(img))

def to_iso(img, is_mask=False, ref=None):
    interp = sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline
    if ref is not None:
        return sitk.Resample(img, ref, sitk.Transform(), interp, 0)
    sp = (1.0, 1.0, 1.0)
    size = [int(round(s * o / n)) for s, o, n in zip(img.GetSize(), img.GetSpacing(), sp)]
    return sitk.Resample(img, size, sitk.Transform(), interp, img.GetOrigin(), sp, img.GetDirection(), 0, img.GetPixelID())

def subject_z(img, brain):
    a = sitk.GetArrayFromImage(img).astype(np.float32); m = sitk.GetArrayFromImage(brain) > 0
    z = np.clip((a - a[m].mean()) / a[m].std(), -3, 3)
    out = sitk.GetImageFromArray(z); out.CopyInformation(img); return out

def preprocess(img, seg):
    """img：該序列的原始影像；seg：已在同一空間（世界座標）的 FreeSurfer aseg。回傳 z 分數影像與 1 mm 等向的分割"""
    img = sitk.Cast(img, sitk.sitkFloat32)
    seg = to_iso(seg, is_mask=True, ref=img)
    img = n4(img, sitk.Cast(seg > 0, sitk.sitkUInt8))
    img = to_iso(img); seg = to_iso(seg, is_mask=True, ref=img)
    return subject_z(img, sitk.Cast(seg > 0, sitk.sitkUInt8)), seg

def register_flair_to_t1(t1, flair):
    """FLAIR → T1 剛體配準（mutual information），只用該受試者自己的兩個影像。
    同一次掃描的兩個序列通常已在相同的掃描儀座標，故從恆等轉換開始，只校正序列間的小幅移動。"""
    fixed, moving = sitk.Cast(t1, sitk.sitkFloat32), sitk.Cast(flair, sitk.sitkFloat32)
    init = sitk.Euler3DTransform()
    init.SetCenter(fixed.TransformContinuousIndexToPhysicalPoint([(n - 1) / 2 for n in fixed.GetSize()]))
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(50)
    R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(0.05, SEED)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetOptimizerAsRegularStepGradientDescent(learningRate=1.0, minStep=1e-4, numberOfIterations=200)
    R.SetOptimizerScalesFromPhysicalShift()
    R.SetShrinkFactorsPerLevel([4, 2, 1]); R.SetSmoothingSigmasPerLevel([2, 1, 0])
    R.SetInitialTransform(init, inPlace=False)
    tx = R.Execute(fixed, moving)
    return tx, R.GetMetricValue()

## M3｜PyRadiomics 萃取與左右不對稱指標（計畫書步驟 0）

設定與計畫書第四段一致：binWidth = 0.05（z 單位）、僅原始影像、七類特徵；形狀特徵只從 T1 取。每個海馬迴萃取四個版本：FreeSurfer aseg（主要，`v0`）、海馬迴亞區模組 HBT（第二套分割，`hbt`；B11，M4 在**訓練摺內**以 aseg 與 HBT 計算 ICC），以及 aseg 侵蝕／膨脹 1 體素（`ero`／`dil`；探索性擾動分析，只報告、不用於篩選）。

每位受試者的結果存成 `subjects/sub-<ID>/features.json`，已完成者重新執行時直接讀取，可分次續跑。M3 只做逐一受試者的運算，所以訓練集與測試集都在這裡萃取；測試集的特徵到 M5 才讀入。


In [ ]:
from radiomics import featureextractor
import logging, yaml; logging.getLogger("radiomics").setLevel(logging.ERROR)

PARAMS = {"setting": {"binWidth": 0.05, "normalize": False, "label": 1, "geometryTolerance": 1e-4},
          "imageType": {"Original": {}},
          "featureClass": {k: [] for k in ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm"]}}
yaml.safe_dump(PARAMS, open(os.path.join(ROOT, "params.yaml"), "w"))

def extractor(no_shape=False, bin_width=None):
    ex = featureextractor.RadiomicsFeatureExtractor(os.path.join(ROOT, "params.yaml"))
    if no_shape: ex.enableFeatureClassByName("shape", enabled=False)       # PyRadiomics API（8ed5793）：以 enabled=False 停用整類
    if bin_width: ex.settings["binWidth"] = bin_width
    return ex

VERSIONS = ["v0", "hbt", "ero", "dil"]   # v0 = aseg；hbt = 海馬迴亞區模組（B11）；ero／dil = aseg 侵蝕／膨脹 1 體素（探索性）

def mask_versions(seg, label, hbt=None):
    roi = sitk.Cast(seg == label, sitk.sitkUInt8)
    mv = {"v0": roi, "ero": sitk.BinaryErode(roi, [1, 1, 1]), "dil": sitk.BinaryDilate(roi, [1, 1, 1])}
    if hbt is not None: mv["hbt"] = sitk.Cast(hbt == label, sitk.sitkUInt8)
    return mv

def _arr(m): return sitk.GetArrayFromImage(m) > 0
def perturb_stats(mv):
    """擾動強度：原始遮罩 vs 侵蝕（v1）／膨脹（v2）的 Dice 與體積變化 %（逐一受試者，不涉及其他人）"""
    a0 = _arr(mv["v0"]); out = {}
    for v in [k for k in ["ero", "dil", "hbt"] if k in mv]:
        a = _arr(mv[v]); out[f"{v}_dice"] = 2 * (a0 & a).sum() / (a0.sum() + a.sum())
        out[f"{v}_vol_change_pct"] = 100 * (a.sum() / a0.sum() - 1)
    return out

def feats(img, roi, ex, prefix):
    r = ex.execute(img, roi, label=1)
    return {f"{prefix}_{k[9:]}": float(v) for k, v in r.items() if k.startswith("original_")}

def asymmetry(row_L, row_R):
    out = {}
    for k, L in row_L.items():
        R = row_R[k.replace("hip_L", "hip_R")]; den = (L + R) / 2
        out[k.replace("hip_L", "hipAI")] = (L - R) / den if abs(den) > 1e-6 else L - R
    return out

if DEMO_MODE:
    import radiomics
    ip, mp = radiomics.getTestCase("brain1")
    demo = feats(sitk.ReadImage(ip), sitk.ReadImage(mp), extractor(bin_width=25), "demo")   # 示範影像未 z 標準化
    print(f"示範萃取成功：{len(demo)} 個特徵")
    print("示範遮罩的擾動強度：", {k: round(v, 3) for k, v in perturb_stats(mask_versions(sitk.ReadImage(mp), 1)).items()})
else:
    exT1, exFL = extractor(), extractor(no_shape=True)

    def extract_subject(pid):
        """逐一受試者：前處理、FLAIR 配準、四個分割版本萃取、左右不對稱指標。結果存在該受試者資料夾，可續跑"""
        d = os.path.join(SUBJ, pid); out = os.path.join(d, "features.json")
        if os.path.exists(out):
            r = json.load(open(out))
            if all(v in r["AI"] for v in VERSIONS): return r                      # B11 之前的舊結果缺 hbt，重算
        t1_raw = sitk.ReadImage(os.path.join(d, "T1w.nii.gz"), sitk.sitkFloat32)
        fl_raw = sitk.ReadImage(os.path.join(d, "FLAIR.nii.gz"), sitk.sitkFloat32)
        aseg = sitk.ReadImage(os.path.join(d, "aseg.nii.gz"))
        hbt = hbt_label_image(d, ref=aseg)                                    # B11：第二套分割，與 aseg 同空間、同標籤編碼
        img, seg = preprocess(t1_raw, aseg)
        hseg = to_iso(hbt, is_mask=True, ref=img)                            # 與 seg 同一個 1 mm 網格
        tx, metric = register_flair_to_t1(t1_raw, fl_raw)
        sitk.WriteTransform(tx, os.path.join(d, "flair_to_t1.tfm"))
        p = np.array(tx.GetParameters())[-6:] if len(tx.GetParameters()) >= 6 else np.zeros(6)
        reg = {"rot_deg": float(np.degrees(np.abs(p[:3])).max()), "trans_mm": float(np.linalg.norm(p[3:])), "mi": float(metric)}
        aseg_fl = sitk.Resample(aseg, fl_raw, tx.GetInverse(), sitk.sitkNearestNeighbor, 0)
        fimg, fseg = preprocess(fl_raw, aseg_fl)
        hbt_fl = sitk.Resample(hbt, fl_raw, tx.GetInverse(), sitk.sitkNearestNeighbor, 0)
        fhseg = to_iso(hbt_fl, is_mask=True, ref=fimg)                       # 與 fseg 同一個 1 mm 網格
        res = {"reg": reg, "pert": {}, "AI": {}}
        mv_t1 = {s: mask_versions(seg, ASEG[s], hseg) for s in ["hip_L", "hip_R"]}
        mv_fl = {s: mask_versions(fseg, ASEG[s], fhseg) for s in ["hip_L", "hip_R"]}
        res["pert"] = {s: perturb_stats(mv_t1[s]) for s in mv_t1}
        for v in VERSIONS:
            side = {s: {**feats(img, mv_t1[s][v], exT1, f"T1_{s}"), **feats(fimg, mv_fl[s][v], exFL, f"FL_{s}")} for s in ["hip_L", "hip_R"]}
            res["AI"][v] = asymmetry(side["hip_L"], side["hip_R"])
        json.dump(res, open(out, "w"))
        return res

    t0 = time.time(); allres = {}
    for k, pid in enumerate(cohort.index, 1):
        t1s = time.time(); allres[pid] = extract_subject(pid)
        print(f"[{k}/{len(cohort)}] {pid}：{time.time() - t1s:.0f} 秒（累計 {(time.time() - t0) / 60:.1f} 分鐘）", flush=True)
    for v in VERSIONS:
        pd.DataFrame({pid: r["AI"][v] for pid, r in allres.items()}).T.rename_axis("pid").to_csv(
            os.path.join(ROOT, f"features_AI_{v}.csv"))                    # 原始版特徵矩陣（四個分割版本）
    pert = pd.DataFrame([{"pid": pid, "side": s, **r["pert"][s]} for pid, r in allres.items() for s in r["pert"]])
    pert.to_csv(os.path.join(ROOT, "perturbation_stats.csv"), index=False)
    reg = pd.DataFrame({pid: r["reg"] for pid, r in allres.items()}).T.rename_axis("pid")
    reg.to_csv(os.path.join(ROOT, "flair_registration.csv"))
    print("擾動強度（中位數）：\n", pert.drop(columns="pid").groupby("side").median().round(3))
    print("FLAIR 配準（中位數與最大值）：\n", reg.agg(["median", "max"]).round(2))


## M4｜只用訓練集：摺內建模、選超參數、定 Youden 切點、定版（計畫書步驟 2–4）

`RadiomicsModel` 在每一摺內部依序做：
1. **ComBat**：只用該摺訓練受試者，以原始遮罩版本估計參數，同一組參數套用到侵蝕、膨脹版本（B3）
2. **ICC(3,1)**：在調和後的三版本**不對稱指標**上計算，< 0.75 者排除（B4）
3. **冗餘移除**：先把 `vol_ai` 視為已保留的參照，與它 |ρ| > 0.90 的影像組學特徵移除；其餘依 ICC 高低保留，彼此 |ρ| ≤ 0.90（B1）
4. **標準化** → 5. **LASSO**（≤ 5 個）→ 6. **最終分類器**

`GridSearchCV(refit=True)` 在選定超參數後以完整訓練集重新擬合一次，即步驟 4 的定版。模型 C 報告用的訓練集 CV AUC 另以**巢狀 CV** 估計（外層 5 摺 × 5 次，內層 5 摺選 C；B2），外層每摺同時記錄 C−A 配對差值與選入特徵（穩定度頻率）。


In [ ]:
META = ["_batch", "_age", "_sex"]; BASE = ["vol_ai", "_age", "_sex"]

if DEMO_MODE:
    # 模擬 200 個不對稱特徵、四個分割版本（aseg、HBT、侵蝕、膨脹）、兩種協定的批次效應、體積不對稱與 HS 病理（僅供演練）
    rng = np.random.default_rng(SEED); n = len(cohort); y = cohort["y"].values; p = 200
    signal = np.zeros(p); signal[:6] = 0.30
    vol_ai = np.where(y == 1, 0.05, -0.05) + rng.normal(0, 0.09, n)
    base = rng.normal(0, 1, (n, p)) + np.outer(np.where(y == 1, -1, 1), signal)
    base[:, 6:26] = base[:, [0]] * 0.95 + rng.normal(0, 0.3, (n, 20))
    base[:, 26:28] = vol_ai[:, None] / 0.09 + rng.normal(0, 0.05, (n, 2))   # 模擬 MeshVolume／VoxelVolume 的不對稱指標（≈ vol_ai）
    batch = rng.integers(0, 2, n); base += np.outer(batch * 0.8, rng.normal(1, 0.3, p))
    noise_sd = np.r_[rng.uniform(0.05, 0.3, p - 40), rng.uniform(0.8, 1.5, 40)]       # 後 40 個特徵不穩定
    cols = [f"hipAI_f{i:03d}" for i in range(p)]
    F = {v: pd.DataFrame(base + (0 if v == "v0" else rng.normal(0, 1, (n, p)) * noise_sd), index=cohort.index, columns=cols)
         for v in ["v0", "hbt", "ero", "dil"]}
    meta = pd.DataFrame({"_batch": batch, "_age": rng.integers(18, 70, n) // 5, "_sex": rng.integers(0, 2, n),
                         "vol_ai": vol_ai,
                         "hs": rng.integers(0, 2, n)}, index=cohort.index)
else:
    F = {v: pd.read_csv(os.path.join(ROOT, f"features_AI_{v}.csv"), index_col=0) for v in ["v0", "hbt", "ero", "dil"]}
    meta = pd.read_csv(os.path.join(ROOT, "meta.csv"), index_col=0)   # _batch（協定群組）、_age、_sex、vol_ai、hs

RAD = list(F["v0"].columns)
X_all = pd.concat([meta[META + ["vol_ai"]], F["v0"]], axis=1)
y_all = cohort["y"]
X_tr, y_tr = X_all.loc[train_ids], y_all.loc[train_ids].values      # M4 只用這兩個
print("訓練集：", X_tr.shape, "事件數 =", int(y_tr.sum()))

In [ ]:
import io, contextlib

def combat_from_training(dat, batch, est):
    """以訓練摺估得的 ComBat 參數調和新資料（dat：特徵 × 受試者）。
    計算與 neuroCombat 0.2.12 的 neuroCombatFromTraining 相同；原函式第 482 行
    int(np.where(...)[0]) 在 NumPy 2.x 會報 TypeError，故在此重寫並改為取 [0][0]。"""
    dat = np.asarray(dat, dtype=float)
    batch = np.array(batch, dtype="str"); old = np.array(est["batches"], dtype="str")
    missing = np.setdiff1d(np.unique(batch), old)
    if missing.size: raise ValueError(f"批次 {missing} 不在訓練摺中")
    wh = [int(np.where(old == b)[0][0]) for b in batch]
    var_pooled = est["var.pooled"]
    stand_mean = est["stand.mean"][:, 0] + est["mod.mean"].mean(axis=1)
    stand_mean = np.transpose([stand_mean] * dat.shape[1])
    z = (dat - stand_mean) / np.sqrt(var_pooled)
    z = (z - np.transpose(est["gamma.star"][wh, :])) / np.sqrt(np.transpose(est["delta.star"][wh, :]))
    return z * np.sqrt(var_pooled) + stand_mean

def icc31(M):
    n, k = M.shape; g = M.mean()
    msr = k * ((M.mean(1) - g) ** 2).sum() / (n - 1); msc = n * ((M.mean(0) - g) ** 2).sum() / (k - 1)
    mse = (((M - g) ** 2).sum() - msr * (n - 1) - msc * (k - 1)) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse)

class RadiomicsModel(ClassifierMixin, BaseEstimator):
    # 摺內：ComBat → ICC → 冗餘移除（參照 vol_ai）→ 標準化 → LASSO（≤ K_MAX）→ 最終分類器
    # use_base=True 為模型 C（加上 BASE 變項），False 為模型 B；final='rf' 為 C-RF
    def __init__(self, C=1.0, use_base=True, final="lr", n_estimators=500, max_depth=None):
        self.C, self.use_base, self.final, self.n_estimators, self.max_depth = C, use_base, final, n_estimators, max_depth
    def _combat_apply(self, Z, batch):
        return combat_from_training(Z.T, batch, self.combat_est_).T
    def _transform_rad(self, X):
        Z = X[RAD].values
        if self.ok_combat_: Z = self._combat_apply(Z, X["_batch"].values)
        Z = pd.DataFrame(Z, columns=RAD, index=X.index)[self.keep_]
        return self.scaler_.transform(Z)[:, self.sel_idx_]
    def fit(self, X, y):
        y = np.asarray(y); self.classes_ = np.unique(y)
        ids = X.index
        V = {v: F[v].loc[ids, RAD] for v in ["v0", "hbt"]}               # 兩套分割的不對稱指標（B11；只取本摺訓練受試者）
        # 1) ComBat（B3）：只用本摺訓練受試者、以 aseg 版本估計參數，同一組參數套用到 aseg 與 HBT 兩個版本
        self.ok_combat_ = False
        if X["_batch"].nunique() > 1:
            try:
                from neuroCombat import neuroCombat
            except ImportError:
                if not DEMO_MODE: raise                                      # 真實資料不允許靜默略過 ComBat
                neuroCombat = None
            if neuroCombat is not None:
                cov = X[["_batch", "_age", "_sex"]].reset_index(drop=True)
                with contextlib.redirect_stdout(io.StringIO()):                # 隱藏 neuroCombat 每次擬合的進度訊息
                    out = neuroCombat(dat=V["v0"].values.T, covars=cov, batch_col="_batch", categorical_cols=["_sex"], continuous_cols=["_age"])
                self.combat_est_, self.ok_combat_ = out["estimates"], True
                # 兩個版本一律用同一個套用函式，避免版本間因處理方式不同而人為降低 ICC，也與驗證／測試資料的處理一致
                for v in ["v0", "hbt"]:
                    V[v] = pd.DataFrame(self._combat_apply(V[v].values, X["_batch"].values), index=ids, columns=RAD)
        # 2) ICC(3,1)（B4、B11）：在調和後兩套分割（aseg、HBT）的「不對稱指標」上計算
        self.icc_ = {c: icc31(np.c_[V["v0"][c], V["hbt"][c]]) for c in RAD}
        self.rad_ = [c for c in RAD if self.icc_[c] >= ICC_MIN]
        Z = V["v0"][self.rad_]
        # 3) 冗餘移除（B1）：vol_ai 視為已保留的參照；其餘依 ICC 高低保留，彼此 |ρ| ≤ CORR_MAX
        order = sorted(self.rad_, key=lambda c: -self.icc_[c])
        rho = pd.concat([X[[REDUNDANCY_REF]], Z[order]], axis=1).corr("spearman").abs()
        keep, self.drop_vol_ = [REDUNDANCY_REF], []
        for c in order:
            if rho.loc[c, REDUNDANCY_REF] > CORR_MAX: self.drop_vol_.append(c); continue
            if all(rho.loc[c, k] <= CORR_MAX for k in keep): keep.append(c)
        self.keep_ = keep[1:]
        if not self.keep_: raise ValueError("ICC 與冗餘篩選後沒有剩下任何影像組學特徵")
        # 4) 標準化（只用本摺統計量）＋ 5) LASSO 選擇，上限 K_MAX
        self.scaler_ = StandardScaler().fit(Z[self.keep_]); Zs = self.scaler_.transform(Z[self.keep_])
        las = LogisticRegression(penalty="l1", C=self.C, solver="liblinear", class_weight="balanced", max_iter=5000).fit(Zs, y)
        coef = las.coef_[0]; nz = np.flatnonzero(coef)
        self.sel_idx_ = nz[np.argsort(-np.abs(coef[nz]))][:K_MAX] if len(nz) else np.array([int(np.argmax(np.abs(coef)))])
        self.selected_ = [self.keep_[i] for i in self.sel_idx_]
        # 6) 最終分類器
        D = Zs[:, self.sel_idx_]
        if self.use_base:
            self.base_scaler_ = StandardScaler().fit(X[BASE]); D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        self.clf_ = (RandomForestClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth, class_weight="balanced", random_state=SEED)
                     if self.final == "rf" else LogisticRegression(class_weight="balanced", max_iter=5000)).fit(D, y)
        return self
    def predict_proba(self, X):
        D = self._transform_rad(X)
        if self.use_base: D = np.c_[self.base_scaler_.transform(X[BASE]), D]
        return self.clf_.predict_proba(D)
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def model_A():   # 基準線：不調參
    from sklearn.compose import ColumnTransformer
    return Pipeline([("pick", ColumnTransformer([("b", "passthrough", BASE)])),
                     ("scale", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])

def cv_summary(gs):
    i = gs.best_index_; return gs.cv_results_["mean_test_score"][i], gs.cv_results_["std_test_score"][i]

# 步驟 2–4
gs_C = GridSearchCV(RadiomicsModel(use_base=True), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_B = GridSearchCV(RadiomicsModel(use_base=False), {"C": LASSO_GRID}, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
gs_RF = GridSearchCV(RadiomicsModel(use_base=True, final="rf", C=gs_C.best_params_["C"]),
                     RF_GRID, scoring="roc_auc", cv=CV, n_jobs=-1, refit=True).fit(X_tr, y_tr)
from sklearn.model_selection import cross_val_score
cvA = cross_val_score(model_A(), X_tr, y_tr, scoring="roc_auc", cv=CV)
final = {"A": model_A().fit(X_tr, y_tr), "B": gs_B.best_estimator_, "C": gs_C.best_estimator_, "C-RF": gs_RF.best_estimator_}
cv_auc = {"A": (cvA.mean(), cvA.std()), "B": cv_summary(gs_B), "C": cv_summary(gs_C), "C-RF": cv_summary(gs_RF)}
cv_auc_C_nonnested = cv_auc["C"]

# B2：模型 C 的訓練集 CV AUC 以巢狀 CV 估計（外層 = 上面同一組 5 摺 × 5 次；內層 5 摺選 C）
# 外層每摺同時記錄：與模型 A 的配對差值（探索性）、選入特徵（穩定度選擇頻率，A1）
import time; t0 = time.time()
nested = []
for i, (tr, va) in enumerate(CV.split(X_tr, y_tr)):
    inner = GridSearchCV(RadiomicsModel(use_base=True), {"C": LASSO_GRID}, scoring="roc_auc",
                         cv=INNER_CV, n_jobs=-1, refit=True).fit(X_tr.iloc[tr], y_tr[tr])
    aC = roc_auc_score(y_tr[va], inner.predict_proba(X_tr.iloc[va])[:, 1])
    aA = roc_auc_score(y_tr[va], model_A().fit(X_tr.iloc[tr], y_tr[tr]).predict_proba(X_tr.iloc[va])[:, 1])
    nested.append({"fold": i, "best_C": inner.best_params_["C"], "AUC_C": aC, "AUC_A": aA, "delta_C_minus_A": aC - aA,
                   "selected": inner.best_estimator_.selected_})
nested = pd.DataFrame(nested)
cv_auc["C"] = (nested["AUC_C"].mean(), nested["AUC_C"].std(ddof=0))
print(f"巢狀 CV 完成，{len(nested)} 個外層摺，耗時 {(time.time() - t0) / 60:.1f} 分鐘")
for k, (m, s) in cv_auc.items(): print(f"模型 {k}：訓練集 CV AUC = {m:.3f} ± {s:.3f}" + ("（巢狀 CV）" if k == "C" else ""))
print(f"（參考）模型 C 非巢狀 CV AUC = {cv_auc_C_nonnested[0]:.3f}，為 30 個 C 中最佳值，偏樂觀，不用於假說判定")
print(f"探索性：外層摺 C−A 配對差值 平均 {nested['delta_C_minus_A'].mean():.3f}，標準差 {nested['delta_C_minus_A'].std(ddof=0):.3f}")
nested.assign(selected=nested["selected"].map(";".join)).to_csv(os.path.join(ROOT, "nested_cv_C.csv"), index=False)

# A1：穩定度選擇頻率（每個外層摺都完整跑一次選擇流程）
fc = final["C"]
freq = pd.Series([c for sel in nested["selected"] for c in sel]).value_counts().div(len(nested))
freq = freq.reindex(freq.index.union(fc.selected_), fill_value=0).sort_values(ascending=False)
freq = pd.DataFrame({"selection_frequency": freq, "in_final_model": freq.index.isin(fc.selected_)})
freq.to_csv(os.path.join(ROOT, "selection_frequency_C.csv"))
print("模型 C 選入特徵（定版）與其穩定度選擇頻率：")
print(freq[freq["in_final_model"]].round(2))

# A3／B5：分割穩定性摘要與「篩選後版」特徵矩陣（定版模型、訓練集）
stab = {"n_features_total": len(RAD), "icc_threshold": ICC_MIN, "icc_form": "ICC(3,1)", "icc_level": "asymmetry index",
        "icc_segmentations": "FreeSurfer aseg vs hippocampal subfield module HBT (B11)", "order": "ComBat -> ICC -> redundancy(ref vol_ai) -> scale -> LASSO",
        "n_pass_icc": len(fc.rad_), "n_removed_as_volume_duplicate": len(fc.drop_vol_), "removed_as_volume_duplicate": fc.drop_vol_,
        "n_after_redundancy": len(fc.keep_), "corr_max": CORR_MAX, "selected": fc.selected_, "combat_applied": fc.ok_combat_}
# 探索性（B11）：aseg／侵蝕／膨脹擾動的 ICC，只報告、不用於篩選；訓練集、未調和原始值
icc_pert = {c: icc31(np.c_[F["v0"].loc[train_ids, c], F["ero"].loc[train_ids, c], F["dil"].loc[train_ids, c]]) for c in RAD}
stab["exploratory_perturbation_icc_n_pass"] = int(sum(v >= ICC_MIN for v in icc_pert.values()))
pd.DataFrame({"ICC31_aseg_vs_hbt_final_model": pd.Series(fc.icc_), "ICC31_perturbation_exploratory": pd.Series(icc_pert)}).to_csv(
    os.path.join(ROOT, "icc_train_final.csv"))
json.dump(stab, open(os.path.join(ROOT, "stability_summary.json"), "w"), indent=1, ensure_ascii=False)
X_tr[fc.keep_].to_csv(os.path.join(ROOT, "features_AI_filtered_train.csv"))       # 篩選後版（未調和原始值）
if DEMO_MODE: F["v0"].to_csv(os.path.join(ROOT, "features_AI_v0.csv"))           # 真實資料在 M3 已存原始版
print({k: v for k, v in stab.items() if k.startswith("n_") or k in ("combat_applied", "exploratory_perturbation_icc_n_pass")})

# 步驟 3：Youden 切點（訓練集樣本外預測，不看測試集）
thr = {}
for k, est in final.items():
    oof = cross_val_predict(clone(est), X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    f, t, th = roc_curve(y_tr, oof); thr[k] = th[np.argmax(t - f)]
print("Youden 切點：", {k: round(v, 3) for k, v in thr.items()})

# 生物學關聯（RQS 第 7 項）：訓練集內，選入特徵在 HS / 非 HS 間的差異，FDR 校正
from scipy.stats import mannwhitneyu
hs = meta.loc[train_ids, "hs"].values; pv = []
for c in final["C"].selected_:
    v = X_tr[c].values; pv.append(mannwhitneyu(v[hs == 1], v[hs == 0]).pvalue)
pv = np.array(pv); order = np.argsort(pv); m = len(pv)
q = np.minimum.accumulate((pv[order] * m / np.arange(1, m + 1))[::-1])[::-1]       # Benjamini–Hochberg
bio = pd.DataFrame({"feature": np.array(final["C"].selected_)[order], "p": pv[order], "q_FDR": np.minimum(q, 1)})
print(bio.round(4))

## M5｜保留測試集只開啟一次（計畫書步驟 5–6）

`open_test_once()` 執行第二次會直接報錯。若在這裡發現問題，**記錄為限制，不回頭修改 M4**。

In [ ]:
_TEST_OPENED = False
def open_test_once():
    global _TEST_OPENED
    if _TEST_OPENED: raise RuntimeError("保留測試集已開啟過；若需再次開啟，必須在報告中揭露並標示為探索性。")
    _TEST_OPENED = True
    return X_all.loc[test_ids], y_all.loc[test_ids].values

def boot(fn, y, *ps, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: out.append(fn(y[i], *[p[i] for p in ps]))
    return np.percentile(out, [2.5, 97.5])

def delong(y, p1, p2):
    def comp(p):
        pos, neg = p[y == 1], p[y == 0]
        return (np.array([np.mean((a > neg) + 0.5 * (a == neg)) for a in pos]),
                np.array([np.mean((pos > b) + 0.5 * (pos == b)) for b in neg]))
    a10, a01 = comp(p1); b10, b01 = comp(p2)
    S = np.cov(np.vstack([a10, b10])) / len(a10) + np.cov(np.vstack([a01, b01])) / len(a01)
    d = a10.mean() - b10.mean(); z = d / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return d, 2 * stats.norm.sf(abs(z))

def calib(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    slope = LogisticRegression(C=1e6).fit(lp.reshape(-1, 1), y).coef_[0, 0]
    inter = brentq(lambda a: np.sum(y - 1 / (1 + np.exp(-(a + lp)))), -10, 10)
    return inter, slope

def net_benefit(y, p, ths):
    return np.array([np.sum((p >= t) & (y == 1)) / len(y) - np.sum((p >= t) & (y == 0)) / len(y) * t / (1 - t) for t in ths])

X_te, y_te = open_test_once()
P = {k: m.predict_proba(X_te)[:, 1] for k, m in final.items()}
rows = []
for k, p in P.items():
    lo, hi = boot(roc_auc_score, y_te, p); inter, slope = calib(y_te, p); pred = p >= thr[k]
    rows.append({"模型": k, "測試 AUC": roc_auc_score(y_te, p), "95% CI": f"{lo:.3f}–{hi:.3f}",
                 "AUPRC": average_precision_score(y_te, p), "敏感度": pred[y_te == 1].mean(), "特異度": (~pred[y_te == 0]).mean(),
                 "校準截距": inter, "校準斜率": slope, "Brier": brier_score_loss(y_te, p),
                 "訓練 CV AUC": cv_auc[k][0], "CV 估計": "巢狀" if k == "C" else ("不調參" if k == "A" else "非巢狀（探索性）"),
                 "CV−測試 差距": cv_auc[k][0] - roc_auc_score(y_te, p)})
res = pd.DataFrame(rows).round(3); display(res)
prev = y_te.mean(); w1, w0 = len(y_tr) / (2 * y_tr.sum()), len(y_tr) / (2 * (len(y_tr) - y_tr.sum()))
print(f"基準線：多數類（左側）準確率 = {1 - prev:.3f}；AUPRC 基準線 = 事件率 = {prev:.3f}")
print(f"註：class_weight='balanced'（右 {w1:.2f}、左 {w0:.2f}）會使預測機率在 logit 尺度偏高約 {np.log(w1 / w0):.2f}，"
      f"校準截距預期約 {-np.log(w1 / w0):.2f}；此為加權所致，於報告中說明，不重新校準。")

# 主要比較（唯一一組）：模型 C vs 模型 A
d, pv = delong(y_te, P["C"], P["A"])
ci = boot(lambda yy, a, b: roc_auc_score(yy, a) - roc_auc_score(yy, b), y_te, P["C"], P["A"])
gap = cv_auc["C"][0] - roc_auc_score(y_te, P["C"])          # B7：單向；正值 = 測試比巢狀 CV 差
print(f"主要比較 ΔAUC(C − A) = {d:.3f}，95% CI {ci[0]:.3f}–{ci[1]:.3f}，DeLong P = {pv:.4f}")
# B6：主要假說三級判定（2026-11-25、開啟測試集前寫定）
if pv < ALPHA and d >= DELTA_AUC_H:
    verdict = f"支持：ΔAUC ≥ {DELTA_AUC_H} 且 DeLong P < {ALPHA}"
elif pv < ALPHA and d > 0:
    verdict = f"有增量但未達預設門檻：DeLong P < {ALPHA}，但 ΔAUC < {DELTA_AUC_H}"
elif pv < ALPHA:
    verdict = f"模型 C 顯著較差：DeLong P < {ALPHA}，ΔAUC < 0（三級之外的情形，照實報告）"
else:
    verdict = f"本樣本無法證明增量：DeLong P ≥ {ALPHA}；以 ΔAUC 與 95% CI 描述效應大小"
print("假說第一條（B6 三級判定）：", verdict)
# B7：第二條為單向判定，只檢查測試是否明顯比巢狀 CV 差
print(f"假說第二條（B7 單向）：巢狀 CV AUC − 測試 AUC = {gap:+.3f}，< {MAX_GAP}？{'是' if gap < MAX_GAP else '否'}")
if gap <= -MAX_GAP:
    print(f"  註：測試 AUC 高於巢狀 CV 達 {-gap:.3f}，第二條仍判通過；於報告中說明此為小樣本測試集的抽樣變異，不解讀為模型變好。")
print("其餘比較（B、C-RF、子群、跨協定、杏仁核）一律標示為探索性。")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, p in P.items():
    f, t, _ = roc_curve(y_te, p); ax[0].plot(f, t, label=f"{k} ({roc_auc_score(y_te, p):.2f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(title="Held-out test ROC (opened once)", xlabel="1 - specificity", ylabel="sensitivity"); ax[0].legend()
from sklearn.calibration import calibration_curve
for k in ["A", "C"]:
    fr, mp = calibration_curve(y_te, P[k], n_bins=5, strategy="quantile"); ax[1].plot(mp, fr, "o-", label=k)
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="Calibration (test)", xlabel="predicted", ylabel="observed"); ax[1].legend()
ths = np.linspace(.05, .9, 40)
for k in ["A", "C"]: ax[2].plot(ths, net_benefit(y_te, P[k], ths), label=k)
ax[2].plot(ths, prev - (1 - prev) * ths / (1 - ths), "k:", label="treat all"); ax[2].axhline(0, color="k", lw=.8)
ax[2].set(title="Decision curve (test)", xlabel="threshold", ylabel="net benefit", ylim=(-.05, prev + .05)); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(ROOT, "fig_test.png"), dpi=200); plt.show()
res.to_csv(os.path.join(ROOT, "test_results.csv"), index=False)
pd.Series(final["C"].selected_).to_csv(os.path.join(ROOT, "selected_features_C.csv"), index=False)
X_te[final["C"].keep_].to_csv(os.path.join(ROOT, "features_AI_filtered_test.csv"))   # 篩選後版（測試集，開啟後才存）

## 切換到真實資料的執行順序與檢查清單

1. M0：`DEMO_MODE = False`。把 figshare tables_metadata 的三個檔放到 Drive 的 `ideas_radiomics/metadata/`。
2. M1a：核對排除流程（預期 442 → 316 → 257 → 257 → 244）。
3. M2a：逐人下載（約 50 GB 流量、Drive 約 4–5 GB）；有錯誤就重新執行本格續傳，直到錯誤 0 人。
4. M2b：看 `volumes_alignment.csv` 的對齊指標與兩套分割的 Dice；逐頁判讀 `qc/qc_page_*.png`，填 `qc_decisions.csv`（收／退）。一週後以 `qc_second_review_sample.csv` 盲審第二次，記錄一致率。
5. M1b：品管完成後才切分；把排除流程人數填入報告並凍結。
6. M3：逐人萃取（可分次續跑）；檢查 `flair_registration.csv` 與 `perturbation_stats.csv`。
7. 產生 `meta.csv`（協定群組 `_batch`、`_age`、`_sex`、`vol_ai`、`hs`）——程式待補，協定分群規則須在建模前寫定。
8. M4 只執行一次完整搜尋；確認 `stability_summary.json` 中 `combat_applied` 為 true。
9. 開啟測試集前：git commit 並打 tag `pre-test-lock`。
10. M5 只執行一次；結果不論好壞都報告。
